In [2]:
import json
from pathlib import Path

import geopandas as gpd
import pandas as pd
import numpy as np

In [ ]:
# CONFIGURAÇÃO DO PROJETO

# Identifica o diretório em que o notebook está sendo executado

CURRENT_DIR = Path.cwd()


# Iidentifica a raiz do projeto

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR


# Localiza o arquivo de configuração

CONFIG_FILE = (
    PROJECT_ROOT
    / "config"
    / "config.json"
)


# Carrega as configurações do projeto

with open(
    CONFIG_FILE,
    "r",
    encoding="utf-8"
) as arquivo:

    config = json.load(arquivo)


print("Raiz do projeto:")
print(PROJECT_ROOT)

Raiz do projeto:
c:\Users\simon\Documents\GitHub\EtPilot


In [5]:
# CAMINHOS

# Utilizo como origem o recorte do Bom Fim, mantendo os destinos distribuídos em Porto Alegre.

ORIGINS_FILE = (
    PROJECT_ROOT
    / config["paths"]["origins_bomfim"]
)

DESTINATIONS_FILE = (
    PROJECT_ROOT
    / config["paths"]["destinations"]
)

OD_PAIRS_FILE = (
    PROJECT_ROOT
    / config["paths"]["od_pairs"]
)


print("Origens:")
print(ORIGINS_FILE)

print("\nDestinos:")
print(DESTINATIONS_FILE)

print("\nPares O-D:")
print(OD_PAIRS_FILE)

Origens:
c:\Users\simon\Documents\GitHub\EtPilot\data\o-d\origens_bomfim.gpkg

Destinos:
c:\Users\simon\Documents\GitHub\EtPilot\data\o-d\destinos_porto_alegre.gpkg

Pares O-D:
c:\Users\simon\Documents\GitHub\EtPilot\data\o-d\od_pairs.gpkg


In [6]:
# ============================================================
# CARREGAMENTO DOS DADOS
# ============================================================

# Carrega as origens selecionadas para o piloto

origens = gpd.read_file(ORIGINS_FILE)


# Carrega os destinos construídos no notebook 06

destinos = gpd.read_file(
    DESTINATIONS_FILE,
    layer="destinations"
)


print(f"Origens: {len(origens):,}")

print(f"Destinos: {len(destinos):,}")

Origens: 11
Destinos: 2,617


In [7]:
print("Colunas das origens:")
print(origens.columns.tolist())

print("\nColunas dos destinos:")
print(destinos.columns.tolist())

Colunas das origens:
['origin_id', 'ID_UNICO', 'nome_1KM', 'nome_5KM', 'nome_10KM', 'nome_50KM', 'nome_100KM', 'nome_500KM', 'QUADRANTE', 'MASC', 'FEM', 'POP', 'DOM_OCU', 'Shape_Leng', 'Shape_Area', 'node', 'geometry']

Colunas dos destinos:
['destination_id', 'category', 'node', 'destination_weight', 'n_subcategories', 'subcategories', 'geometry']


In [8]:
# Verifica se as colunas essenciais dos destinos estão disponíveis.

colunas_destinos = {
    "destination_id",
    "category",
    "node",
    "destination_weight",
    "geometry"
}


faltantes_destinos = (
    colunas_destinos
    -
    set(destinos.columns)
)


if faltantes_destinos:

    raise ValueError(
        "Colunas ausentes nos destinos: "
        f"{faltantes_destinos}"
    )


print(
    "✓ Estrutura dos destinos válida."
)

✓ Estrutura dos destinos válida.


In [10]:
origens.head()
origens.columns.tolist()

['origin_id',
 'ID_UNICO',
 'nome_1KM',
 'nome_5KM',
 'nome_10KM',
 'nome_50KM',
 'nome_100KM',
 'nome_500KM',
 'QUADRANTE',
 'MASC',
 'FEM',
 'POP',
 'DOM_OCU',
 'Shape_Leng',
 'Shape_Area',
 'node',
 'geometry']

In [11]:
# Manter o identificador original da origem e renomeia o nó para deixar explícito que se trata do nó de origem.

origens = origens.rename(
    columns={
        "node": "origin_node",
        "POP": "origin_weight",
        "MASC": "male_population",
        "FEM": "female_population"
    }
)

In [12]:
# Seleciona os atributos necessários para construir os pares origem-destino

origens_od = origens[
    [
        "origin_id",
        "ID_UNICO",
        "origin_node",
        "origin_weight",
        "male_population",
        "female_population"
    ]
].copy()

In [13]:
# VALIDAÇÃO DAS ORIGENS

print(f"Número de origens: {len(origens_od):,}")

print(
    "Origin IDs duplicados:",
    origens_od["origin_id"]
    .duplicated()
    .sum()
)

print(
    "Nós ausentes:",
    origens_od["origin_node"]
    .isna()
    .sum()
)

print(
    "População ausente:",
    origens_od["origin_weight"]
    .isna()
    .sum()
)

print(
    "População total:",
    origens_od["origin_weight"]
    .sum()
)

Número de origens: 11
Origin IDs duplicados: 0
Nós ausentes: 0
População ausente: 0
População total: 11151.0


In [15]:
# Verifica se a população total corresponde a soma das populações masculina e feminina

origens_od["population_check"] = (
    origens_od["male_population"]
    +
    origens_od["female_population"]
)


origens_od["population_difference"] = (
    origens_od["origin_weight"]
    -
    origens_od["population_check"]
)


origens_od[
    origens_od["population_difference"] != 0
][
    [
        "origin_id",
        "origin_weight",
        "male_population",
        "female_population",
        "population_difference"
    ]
].head(20)

origens_od = origens_od.drop(
    columns=[
        "population_check",
        "population_difference"
    ],
    errors="ignore"
)

In [ ]:
# PREPARAÇÃO DOS DESTINOS

destinos = destinos.rename(
    columns={
        "node": "destination_node"
    }
)


destinos_od = destinos[
    [
        "destination_id",
        "destination_node",
        "category",
        "destination_weight"
    ]
].copy()



,origin_id,ID_UNICO,origin_node,origin_weight,male_population,female_population
0,2835.0,200ME52788N80052,295523697,1113.0,487.0,626.0
1,2839.0,200ME52786N80054,457564753,1229.0,550.0,679.0
2,2840.0,200ME52788N80054,686599378,1091.0,463.0,628.0
3,3437.0,200ME52792N80050,295523686,1264.0,481.0,783.0
4,3438.0,200ME52794N80050,330047229,747.0,323.0,424.0


In [18]:
origens_od.head()

,origin_id,ID_UNICO,origin_node,origin_weight,male_population,female_population
0,2835.0,200ME52788N80052,295523697,1113.0,487.0,626.0
1,2839.0,200ME52786N80054,457564753,1229.0,550.0,679.0
2,2840.0,200ME52788N80054,686599378,1091.0,463.0,628.0
3,3437.0,200ME52792N80050,295523686,1264.0,481.0,783.0
4,3438.0,200ME52794N80050,330047229,747.0,323.0,424.0


In [19]:
destinos_od.head()

,destination_id,destination_node,category,destination_weight
0,EDU_00001,293823357,education,1
1,EDU_00002,293823550,education,1
2,EDU_00003,294899675,education,1
3,EDU_00004,295145441,education,1
4,EDU_00005,295165445,education,1


In [20]:
# DIMENSÃO DO PRODUTO

n_origens = len(
    origens_od
)

n_destinos = len(
    destinos_od
)

n_pares_teorico = (
    n_origens
    *
    n_destinos
)


print(f"Origens: {n_origens:,}")

print(f"Destinos: {n_destinos:,}")

print(f"Pares O-D possíveis: {n_pares_teorico:,}")

Origens: 11
Destinos: 2,617
Pares O-D possíveis: 28,787


In [21]:
pares_por_categoria = (
    destinos_od
    .groupby("category")
    .size()
    .rename("destinations")
    .reset_index()
)


pares_por_categoria["origins"] = (
    n_origens
)


pares_por_categoria["od_pairs"] = (
    pares_por_categoria["destinations"]
    *
    pares_por_categoria["origins"]
)


pares_por_categoria

,category,destinations,origins,od_pairs
0,education,480,11,5280
1,health,158,11,1738
2,leisure,919,11,10109
3,shopping,1060,11,11660
